##libraries

In [ ]:
import pandas as pd
import requests
import os


In [ ]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'}
base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_"

## check how many files are required to make up 30 GB

In [ ]:
target_gb = 30
target_bytes = target_gb * 1024 * 1024 * 1024

start_date = '2009-01-01'
end_date = '2025-12-31'
months_range = pd.date_range(start=start_date, end=end_date, freq='MS')
months_to_process = [d.strftime('%Y-%m') for d in months_range]

monthly_file_info = []
total_accumulated_bytes = 0

print(f"Searching for files to reach {target_gb} GB target...")

for month in months_to_process:
    if total_accumulated_bytes >= target_bytes:
        break

    file_url = f"{base_url}{month}.parquet"
    try:
        response = requests.get(file_url, headers=headers, stream=True, timeout=10)
        if response.status_code == 200:
            file_size = int(response.headers.get('Content-Length', 0))
            total_accumulated_bytes += file_size
            monthly_file_info.append({'month': month, 'size_mb': file_size / (1024**2)})
            print(f"Found {month}: {file_size / (1024**2):.2f} MB | Total: {total_accumulated_bytes / (1024**3):.2f} GB")
    except:
        continue

print(f"\nTotal Months discovered: {len(monthly_file_info)}")
print(f"Total Size: {total_accumulated_bytes / (1024**3):.2f} GB")

Searching for files to reach 30 GB target...
Found 2009-01: 448.00 MB | Total: 0.44 GB
Found 2009-02: 422.89 MB | Total: 0.85 GB
Found 2009-03: 460.24 MB | Total: 1.30 GB
Found 2009-04: 455.98 MB | Total: 1.75 GB
Found 2009-05: 472.38 MB | Total: 2.21 GB
Found 2009-06: 451.65 MB | Total: 2.65 GB
Found 2009-07: 433.55 MB | Total: 3.07 GB
Found 2009-08: 437.04 MB | Total: 3.50 GB
Found 2009-09: 446.65 MB | Total: 3.93 GB
Found 2009-10: 503.09 MB | Total: 4.43 GB
Found 2009-11: 456.15 MB | Total: 4.87 GB
Found 2009-12: 465.00 MB | Total: 5.32 GB
Found 2010-01: 491.53 MB | Total: 5.80 GB
Found 2010-02: 342.10 MB | Total: 6.14 GB
Found 2010-03: 389.78 MB | Total: 6.52 GB
Found 2010-04: 503.83 MB | Total: 7.01 GB
Found 2010-05: 516.69 MB | Total: 7.52 GB
Found 2010-06: 493.55 MB | Total: 8.00 GB
Found 2010-07: 487.32 MB | Total: 8.47 GB
Found 2010-08: 378.74 MB | Total: 8.84 GB
Found 2010-09: 484.66 MB | Total: 9.32 GB
Found 2010-10: 452.64 MB | Total: 9.76 GB
Found 2010-11: 427.63 MB | Tota

## dataset familiarization

### sample dataset to familiarize

In [ ]:
# URL for a single month to inspect
inspect_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
sample_file = "sample_inspection.parquet"

#sample file
response = requests.get(inspect_url)
with open(sample_file, 'wb') as f:
    f.write(response.content)

# Load the data
df_sample = pd.read_parquet(sample_file)

# schema and data types
print("\nData Info (Columns & Types)")
print(df_sample.info())

# few rows of data
print("\nData Preview (First 5 rows)")
display(df_sample.head())

# descriptive statistics for numeric columns
print("\nSummary Statistics")
display(df_sample.describe())


Data Info (Columns & Types)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2964624 entries, 0 to 2964623
Data columns (total 19 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64      

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0



Summary Statistics


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
count,2.964624e+06,2964624,2964624,2.824462e+06,2.964624e+06,2.824462e+06,2.964624e+06,2.964624e+06,2.964624e+06,2.964624e+06,2.964624e+06,2.964624e+06,2.964624e+06,2.964624e+06,2.964624e+06,2.964624e+06,2.824462e+06,2.824462e+06
mean,1.754204e+00,2024-01-17 00:46:36.431092,2024-01-17 01:02:13.208130,1.339281e+00,3.652169e+00,2.069359e+00,1.660179e+02,1.651167e+02,1.161271e+00,1.817506e+01,1.451598e+00,4.833823e-01,3.335870e+00,5.270212e-01,9.756319e-01,2.680150e+01,2.256122e+00,1.411611e-01
min,1.000000e+00,2002-12-31 22:59:39,2002-12-31 23:05:41,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,-8.990000e+02,-7.500000e+00,-5.000000e-01,-8.000000e+01,-8.000000e+01,-1.000000e+00,-9.000000e+02,-2.500000e+00,-1.750000e+00
25%,2.000000e+00,2024-01-09 15:59:19.750000,2024-01-09 16:16:23,1.000000e+00,1.000000e+00,1.000000e+00,1.320000e+02,1.140000e+02,1.000000e+00,8.600000e+00,0.000000e+00,5.000000e-01,1.000000e+00,0.000000e+00,1.000000e+00,1.538000e+01,2.500000e+00,0.000000e+00
50%,2.000000e+00,2024-01-17 10:45:37.500000,2024-01-17 11:03:51.500000,1.000000e+00,1.680000e+00,1.000000e+00,1.620000e+02,1.620000e+02,1.000000e+00,1.280000e+01,1.000000e+00,5.000000e-01,2.700000e+00,0.000000e+00,1.000000e+00,2.010000e+01,2.500000e+00,0.000000e+00
75%,2.000000e+00,2024-01-24 18:23:52.250000,2024-01-24 18:40:29,1.000000e+00,3.110000e+00,1.000000e+00,2.340000e+02,2.340000e+02,1.000000e+00,2.050000e+01,2.500000e+00,5.000000e-01,4.120000e+00,0.000000e+00,1.000000e+00,2.856000e+01,2.500000e+00,0.000000e+00
max,6.000000e+00,2024-02-01 00:01:15,2024-02-02 13:56:52,9.000000e+00,3.127223e+05,9.900000e+01,2.650000e+02,2.650000e+02,4.000000e+00,5.000000e+03,1.425000e+01,4.000000e+00,4.280000e+02,1.159200e+02,1.000000e+00,5.000000e+03,2.500000e+00,1.750000e+00
std,4.325902e-01,NaN,NaN,8.502817e-01,2.254626e+02,9.823219e+00,6.362391e+01,6.931535e+01,5.808686e-01,1.894955e+01,1.804102e+00,1.177600e-01,3.896551e+00,2.128310e+00,2.183645e-01,2.338558e+01,8.232747e-01,4.876239e-01


### check possible night time proportion of records

In [ ]:

month_target = "2014-01"
file_url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{month_target}.parquet"
temp_file = f"temp_{month_target}_time.parquet"

print(f"Downloading {month_target} data for day/night analysis.")
response = requests.get(file_url)
with open(temp_file, 'wb') as f:
    f.write(response.content)

# Load only the pickup datetime column
df_time = pd.read_parquet(temp_file, columns=['tpep_pickup_datetime'])

# Extract the hour
hour = df_time['tpep_pickup_datetime'].dt.hour

# Define Day (6am to 6pm) vs Night
# Day: 6, 7, ..., 17 | Night: 18, 19, ..., 5
df_time['time_of_day'] = 'Night'
df_time.loc[(hour >= 6) & (hour < 18), 'time_of_day'] = 'Day'

# Calculate counts
time_counts = df_time['time_of_day'].value_counts()

print(f"\nDay vs. Night Trip Counts for {month_target}")
print(time_counts)
print(f"\nPercentage Day: {(time_counts['Day'] / len(df_time) * 100):.2f}%")

# Cleanup
os.remove(temp_file)
del df_time


Day vs. Night Trip Counts for 2014-01
time_of_day
Day      7364425
Night    6418092
Name: count, dtype: int64

Percentage Day: 53.43%


### check payment type distribution

In [ ]:

month_target = "2014-01"
file_url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{month_target}.parquet"
temp_file = f"temp_{month_target}_payment.parquet"

print(f"Downloading {month_target} data for payment analysis.")
response = requests.get(file_url)
with open(temp_file, 'wb') as f:
    f.write(response.content)

df_pay = pd.read_parquet(temp_file, columns=['payment_type'])

payment_map = {
    1: 'Credit Card',
    2: 'Cash',
    3: 'No Charge',
    4: 'Dispute',
    5: 'Unknown',
    6: 'Void'
}
df_pay['payment_label'] = df_pay['payment_type'].map(payment_map)


payment_counts = df_pay['payment_label'].value_counts()

print(f"\nPayment Type Distribution for {month_target}")
print(payment_counts)

os.remove(temp_file)
del df_pay


Payment Type Distribution for 2014-01
payment_label
Credit Card    7855148
Cash           5818066
Unknown          67880
No Charge        32076
Dispute           9347
Name: count, dtype: int64


### check trips per passenger count

In [ ]:
month_target = "2014-01"
file_url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{month_target}.parquet"
temp_file = f"temp_{month_target}_passengers.parquet"

print(f"Downloading {month_target} data for passenger count analysis.")
response = requests.get(file_url)
with open(temp_file, 'wb') as f:
    f.write(response.content)

# Load the passenger_count column
df_pass = pd.read_parquet(temp_file, columns=['passenger_count'])

# Calculate frequency of each passenger count
passenger_dist = df_pass['passenger_count'].value_counts().sort_index()

print(f"\nPassenger Count Distribution for {month_target}")
print(passenger_dist)

# Cleanup
os.remove(temp_file)
del df_pass


Passenger Count Distribution for 2014-01
passenger_count
0          259
1      9727321
2      1891588
3       566248
4       267540
5       789070
6       540444
7            7
8            5
9           16
208         19
Name: count, dtype: int64


###check RateCode

In [ ]:
month_target = "2014-01"
file_url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{month_target}.parquet"
temp_file = f"temp_{month_target}_ratecode.parquet"

print(f"Downloading {month_target} data for rate code analysis.")
response = requests.get(file_url)
with open(temp_file, 'wb') as f:
    f.write(response.content)


df_rate = pd.read_parquet(temp_file, columns=['RatecodeID'])

rate_map = {
    1.0: 'Standard Rate',
    2.0: 'JFK',
    3.0: 'Newark',
    4.0: 'Nassau/Westchester',
    5.0: 'Negotiated Fare',
    6.0: 'Group Ride'
}

df_rate['rate_label'] = df_rate['RatecodeID'].map(rate_map)
rate_counts = df_rate['rate_label'].value_counts()

print(f"\nRate Code Distribution for {month_target}")
print(rate_counts)

os.remove(temp_file)
del df_rate


Rate Code Distribution for 2014-01
rate_label
Standard Rate         13490929
JFK                     227834
Negotiated Fare          38130
Newark                   19380
Nassau/Westchester        4550
Group Ride                 145
Name: count, dtype: int64


##download .parquet files

In [ ]:

# Configuration
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'}
base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_"

# Define the range for 2024 and 2025
years = [2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025]
months = range(1, 13)

print("Starting extraction for 2014 and 2025 data...")

for year in years:
    for month in months:
        month_str = f"{year}-{month:02d}"
        file_name = f"yellow_tripdata_{month_str}.parquet"
        file_url = f"{base_url}{month_str}.parquet"

        try:
            # Check if file already exists
            if os.path.exists(file_name):
                print(f"File {file_name} already exists. Skipping.")
                continue

            # Attempt to download
            response = requests.get(file_url, headers=headers, stream=True)

            if response.status_code == 200:
                with open(file_name, 'wb') as f:
                    f.write(response.content)
                print(f"Successfully saved: {file_name} ({len(response.content)/(1024**2):.2f} MB)")
            elif response.status_code == 403 or response.status_code == 404:
                # This is expected for future months not yet released by TLC
                pass
            else:
                print(f"Could not download {month_str}: Status {response.status_code}")

        except Exception as e:
            print(f"Error processing {month_str}: {e}")

print("\nProcess complete. Check the file explorer on the left to see your saved files.")

Starting extraction for 2014 and 2025 data...
Successfully saved: yellow_tripdata_2014-01.parquet (164.01 MB)
Successfully saved: yellow_tripdata_2014-02.parquet (154.69 MB)
Successfully saved: yellow_tripdata_2014-03.parquet (183.52 MB)
Successfully saved: yellow_tripdata_2014-04.parquet (174.71 MB)
Successfully saved: yellow_tripdata_2014-05.parquet (178.05 MB)
Successfully saved: yellow_tripdata_2014-06.parquet (165.78 MB)
Successfully saved: yellow_tripdata_2014-07.parquet (157.46 MB)
Successfully saved: yellow_tripdata_2014-08.parquet (165.17 MB)
Successfully saved: yellow_tripdata_2014-09.parquet (175.34 MB)
Successfully saved: yellow_tripdata_2014-10.parquet (186.59 MB)
Successfully saved: yellow_tripdata_2014-11.parquet (172.70 MB)
Successfully saved: yellow_tripdata_2014-12.parquet (170.71 MB)
Successfully saved: yellow_tripdata_2015-01.parquet (167.20 MB)
Successfully saved: yellow_tripdata_2015-02.parquet (163.69 MB)
Successfully saved: yellow_tripdata_2015-03.parquet (176.7

## check which years selected would make up around 12GB+

In [ ]:
import glob
import os

# Define the years requested
years = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

all_files = []
for year in years:
    # Find all monthly parquet files for the specific year
    all_files.extend(glob.glob(f'yellow_tripdata_{year}-*.parquet'))

# Calculate total size
total_bytes = sum(os.path.getsize(f) for f in all_files)
total_gb = total_bytes / (1024**3)
total_mb = total_bytes / (1024**2)

print(f"Number of files found: {len(all_files)}")
print(f"Years included: {min(years)} to {max(years)}")
print(f"Total combined size: {total_mb:.2f} MB ({total_gb:.2f} GB)")

Number of files found: 0
Years included: 2014 to 2025
Total combined size: 0.00 MB (0.00 GB)


## download the files for taxi trips that took place between 20:00pm till 5:00am

In [ ]:
all_local_files = glob.glob('yellow_tripdata_*.parquet')

for file_path in all_local_files:
    try:

        df = pd.read_parquet(file_path)


        pickup_col = 'tpep_pickup_datetime' if 'tpep_pickup_datetime' in df.columns else 'Trip_Pickup_DateTime'

        if pickup_col not in df.columns:
            print(f"Warning: No datetime column found in {file_path}. Skipping.")
            continue


        df[pickup_col] = pd.to_datetime(df[pickup_col])

        hours = df[pickup_col].dt.hour
        night_df = df[(hours >= 20) | (hours < 5)].copy()


        night_df.to_parquet(file_path, index=False)

        reduction = (1 - len(night_df)/len(df)) * 100
        print(f"Processed {file_path}: Kept {len(night_df):,} trips (Reduced by {reduction:.1f}%)")

        del df
        del night_df

    except Exception as e:
        print(f"Error processing {file_path}: {e}")

print("\nFiltering complete. All files now contain only night-time trips.")

In [ ]:
import glob
from google.colab import files

parquet_files = glob.glob('yellow_tripdata_*.parquet')

if not parquet_files:
    print("No parquet files found to download.")
else:
    print(f"Starting download for {len(parquet_files)} files...")
    for file_path in sorted(parquet_files):
        print(f"Downloading: {file_path}")
        files.download(file_path)